# Intelligence Agent — user perspective

**What it does.** Provides two prompt builders that the Work Item Orchestrator runs inline within its single SDK session:

- `build_enrichment_prompt(work_item_id, work_item_json)` — WIO Step 1 (skill 10): retrieve relevant Knowledge Store entries before Builder runs.
- `build_storage_prompt(qa_result, implementation_notes)` — WIO Step 5 (skill 11): evaluate QA findings against the ingestion criteria and write new Knowledge Store entries.

**Not a standalone agent.** There is no `run_intelligence_agent()` — Intelligence is a function library. The LLM that executes the prompt is the same WIO `ClaudeSDKClient` session (D-04).

**Cost.** This notebook only assembles prompt strings — no LLM call, no Linear, no flags.

In [ ]:
import sys
from pathlib import Path

# Robust _helpers.py discovery — works whether jupyter lab was launched from
# the repo root, notebooks/, or notebooks/agents/. The loop walks up from cwd
# until it finds the parent that holds _helpers.py and inserts it on sys.path.
nb_dir = Path.cwd()
for _parent in (nb_dir, *nb_dir.parents):
    if (_parent / "_helpers.py").exists():
        sys.path.insert(0, str(_parent))
        break

from _helpers import ensure_src_on_path, runtime_summary

ROOT = ensure_src_on_path()
print("HSB_RUNTIME_<AGENT> selection:\n" + runtime_summary())

## Enrichment prompt (WIO Step 1, skill 10)

Returned string is what the WIO sends as a `client.query(...)` turn before Builder runs. It instructs the LLM to use `Glob`/`Grep` over `knowledge/` and produce a `knowledge_context` payload.

In [ ]:
import json

from hsb.agents.intelligence_agent import build_enrichment_prompt

work_item = {
    "id": "LIN-123",
    "title": "Add Postgres optimistic-lock retry",
    "description": "Linear writes are silently overwritten under concurrent claims",
    "labels": ["postgres", "concurrency"],
}
prompt = build_enrichment_prompt("LIN-123", json.dumps(work_item, indent=2))
print(prompt[:600], "..." if len(prompt) > 600 else "")

## Storage prompt (WIO Step 5, skill 11)

Built after QA returns. Instructs the LLM to apply the skill 11 ingestion criteria and write entries to `knowledge/<category>/` only when the signal threshold is met.

In [ ]:
from hsb.agents.intelligence_agent import build_storage_prompt

qa_result = {
    "qa_status": "approved",
    "qa_cycle_count": 2,
    "findings": [{"category": "architecture", "summary": "retry policy uncentralised"}],
}
notes = {"decisions": ["used updatedAt re-read"], "risks": ["no idempotency token yet"]}
prompt = build_storage_prompt(qa_result, notes)
print(prompt[:600], "..." if len(prompt) > 600 else "")

## Air-gap (INTL-04)

The Intelligence module must never import the Linear Agent — Linear writes happen via the WIO's MCP tool wrapper, never via this module. Quick sanity check:

In [ ]:
import re

src = (ROOT / "src/hsb/agents/intelligence_agent.py").read_text()
# Substring `linear_agent` legitimately appears in the docstring's prose
# ("this module never imports from hsb.agents.linear_agent"), so check for
# real `from`/`import` statements specifically. Match against the full source
# (not per-line) so parenthesized multi-line imports like
# `from hsb.agents import (\n    linear_agent,\n)` are also caught.
import_pattern = re.compile(
    r"^[ \t]*(?:from|import)[ \t]+"
    r"(?:[^\n(]*\blinear_agent\b[^\n]*"  # single-line form
    r"|\S+[ \t]+import[ \t]*\([^)]*\blinear_agent\b[^)]*\))",  # parenthesized form
    re.MULTILINE,
)
import_lines = import_pattern.findall(src)
assert not import_lines, f"INTL-04 violation: {import_lines}"
print("INTL-04 OK — no linear_agent import in intelligence_agent.py")